
# Assignment 1 – Web Scraping & Data Analysis (TVMaze)

This notebook:

1. Scrapes at least **200** TV shows from the **TVMaze API**  
2. Cleans and saves the dataset to `your_name+id.csv`  
3. Explores the data with **6+ figures** using **matplotlib**

> Notes
> - Internet access is required to run the scraping cell (uses `requests`).
> - No external keys needed (public API).
> - Figures: use **matplotlib**, **one chart per figure**, and **do not set specific colors**.


In [ ]:

# === Configuration (Edit me) ===
NAME = "AndyLau"   # <- replace with your name (no spaces if possible)
ID = "000000"      # <- replace with your student ID
OUTPUT_CSV = f"{NAME}+{ID}.csv"

# How many shows do we want to collect (minimum 200 as per assignment)?
TARGET_ROWS = 250

# Reproducibility / options
SHOW_PREVIEW_ROWS = 5

print("Output CSV will be:", OUTPUT_CSV)


In [ ]:

# === Imports ===
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime
import math

import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 200)


## 1. Web Scraping

In [ ]:

# TVMaze shows endpoint uses paging starting from 0
# https://www.tvmaze.com/api#show-index
# Each page can contain ~250 shows; we'll loop pages until we hit TARGET_ROWS

def fetch_shows(target_rows=250, max_pages=20):
    records = []
    page = 0
    while len(records) < target_rows and page < max_pages:
        url = f"https://api.tvmaze.com/shows?page={page}"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        shows = r.json()
        for s in shows:
            # Safely extract nested fields
            rating = None
            if isinstance(s.get("rating"), dict):
                rating = s.get("rating", {}).get("average")

            network_name = None
            if isinstance(s.get("network"), dict) and s.get("network"):
                network_name = s["network"].get("name")
            elif isinstance(s.get("webChannel"), dict) and s.get("webChannel"):
                # Use webChannel as fallback
                network_name = s["webChannel"].get("name")

            records.append({
                "Title": s.get("name"),
                "First air date": s.get("premiered"),
                "End date": s.get("ended"),
                "Rating": rating,
                "Genres": s.get("genres", []),
                "Status": s.get("status"),
                "Network": network_name,
                "Summary": s.get("summary")
            })

        page += 1

    return records[:target_rows]

raw_records = fetch_shows(target_rows=TARGET_ROWS)
df_raw = pd.DataFrame(raw_records)
print("Raw shape:", df_raw.shape)
df_raw.head(SHOW_PREVIEW_ROWS)


## 2. Data Cleaning

In [ ]:

def strip_html(text):
    if text is None:
        return None
    # remove HTML tags
    text = re.sub(r"<[^>]+>", "", str(text))
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.copy()

# Clean summary
df["Summary"] = df["Summary"].apply(strip_html)

# Ensure dates are parsed; create helper columns
def to_date(x):
    try:
        return pd.to_datetime(x, errors="coerce").date()
    except Exception:
        return pd.NaT

df["First air date"] = pd.to_datetime(df["First air date"], errors="coerce")
df["End date"] = pd.to_datetime(df["End date"], errors="coerce")

# Premiere year for trend analysis
df["Premiere Year"] = df["First air date"].dt.year

# Run length in days (if ended)
df["Run Length (days)"] = (df["End date"] - df["First air date"]).dt.days

# Replace empty lists of genres with None for clarity, and ensure list type
df["Genres"] = df["Genres"].apply(lambda g: g if isinstance(g, list) and len(g) > 0 else None)

# Drop duplicates by Title + First air date (conservative)
df.drop_duplicates(subset=["Title", "First air date"], inplace=True)

print("Cleaned shape:", df.shape)
df.head(SHOW_PREVIEW_ROWS)


## 3. Save to CSV

In [ ]:

# Keep only required columns (assignment requires these)
required_cols = ["Title", "First air date", "End date", "Rating", "Genres", "Status", "Network", "Summary"]
out_df = df[required_cols].copy()

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(out_df)} rows to {OUTPUT_CSV}")
out_df.head(SHOW_PREVIEW_ROWS)



## 4. Exploratory Data Analysis (EDA)

We answer: **What patterns in genres, platforms, time, and status relate to ratings?**  
We keep visualizations **simple and readable** (matplotlib, one chart per figure).


In [ ]:

# Figure 1 — Rating distribution
plt.figure()
valid_ratings = out_df["Rating"].dropna()
plt.hist(valid_ratings, bins=20)
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()


In [ ]:

# Figure 2 — Top 15 Genres by Count
from collections import Counter

genre_counter = Counter()
for gs in out_df["Genres"].dropna():
    for g in gs:
        genre_counter[g] += 1

genre_items = sorted(genre_counter.items(), key=lambda x: x[1], reverse=True)[:15]
labels = [k for k, v in genre_items]
values = [v for k, v in genre_items]

plt.figure()
plt.bar(range(len(values)), values)
plt.xticks(range(len(values)), labels, rotation=45, ha="right")
plt.title("Top 15 Genres by Show Count")
plt.xlabel("Genre")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:

# Helper: explode genres for per-genre stats
def explode_genres(df_in):
    # Returns a DataFrame with one row per (show, genre)
    rows = []
    for _, r in df_in.iterrows():
        gs = r["Genres"] if isinstance(r["Genres"], list) else []
        for g in gs:
            rows.append({
                "Title": r["Title"],
                "Genre": g,
                "Rating": r["Rating"],
                "Network": r["Network"],
                "Premiere Year": r.get("Premiere Year"),
                "Status": r.get("Status")
            })
    return pd.DataFrame(rows)

gdf = explode_genres(df)
gdf.head(3)


In [ ]:

# Figure 3 — Average Rating by Genre (Top 12 by count)
genre_counts = gdf["Genre"].value_counts()
top_genres = genre_counts.head(12).index
avg_by_genre = gdf[gdf["Genre"].isin(top_genres)].groupby("Genre")["Rating"].mean().sort_values(ascending=False)

plt.figure()
plt.bar(range(len(avg_by_genre)), avg_by_genre.values)
plt.xticks(range(len(avg_by_genre)), avg_by_genre.index, rotation=45, ha="right")
plt.title("Average Rating by Genre (Top 12 Genres)")
plt.xlabel("Genre")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()


In [ ]:

# Figure 4 — Average Rating by Network (Top 12 networks by count)
net_counts = out_df["Network"].value_counts().dropna()
top_nets = net_counts.head(12).index
avg_by_net = out_df[out_df["Network"].isin(top_nets)].groupby("Network")["Rating"].mean().sort_values(ascending=False)

plt.figure()
plt.bar(range(len(avg_by_net)), avg_by_net.values)
plt.xticks(range(len(avg_by_net)), avg_by_net.index, rotation=45, ha="right")
plt.title("Average Rating by Network (Top 12 by Count)")
plt.xlabel("Network")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()


In [ ]:

# Figure 5 — Average Rating by Premiere Year
yearly = df.dropna(subset=["Premiere Year"]).groupby("Premiere Year")["Rating"].mean().dropna()

plt.figure()
plt.plot(yearly.index, yearly.values, marker="o")
plt.title("Average Rating by Premiere Year")
plt.xlabel("Premiere Year")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()


In [ ]:

# Figure 6 — Status vs Rating (Boxplot)
subset = out_df.dropna(subset=["Rating", "Status"])
groups = [subset[subset["Status"] == s]["Rating"].values for s in subset["Status"].dropna().unique()]
labels = list(subset["Status"].dropna().unique())

plt.figure()
plt.boxplot(groups, labels=labels, showmeans=True)
plt.title("Rating by Show Status")
plt.xlabel("Status")
plt.ylabel("Rating")
plt.tight_layout()
plt.show()



## 5. Findings & Discussion (Fill in your story)

- Which **genres** tend to have higher ratings?
- Do certain **networks/platforms** consistently deliver higher-rated shows?
- Are newer shows **improving or declining** in average rating over time?
- Does **status** (Ended vs Running) relate to higher ratings? Why might that be?  
  *(e.g., shows that ended may have had complete arcs; survivorship bias; etc.)*

Use the charts to craft a coherent narrative. Cite specific statistics you observe from your dataset.



## 6. Conclusion

Summarize your main insights and propose future work (e.g., incorporate **cast size**, **episode count**, or **country** to refine conclusions).
